# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Youssof-Essam/Flyrank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — Finding #1: The Anatomy of Growing Content (CONFIRMED)

**Paper claim:** Growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days) than declining content. Sample: 74,187 rising vs 45,272 falling.

**Where does the label come from?** Trend Direction = 30-day vs previous-30-day impression change within the trailing 90-day window. Buckets: Up (>+10%), Down (<-10%), Stable (within ±10%), Flat, New. This is a **current-window proxy** computed from `trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100` — both inputs sit inside the same 90-day aggregate window used for features like `impressions_90d`.

**Methodology question (constructive):** The label and several features share the same 90-day measurement window. For example `impressions_90d` sums the entire 90 days that also contains `impressions_last_30d` and `impressions_prev_30d`. This is not hidden — the paper discloses the 90-day window and the 30-day trend — but it means the comparison is **observational and contemporaneous**, not predictive. If we used this label as a supervised target with 90-day aggregates as features, we would be learning a contemporaneous description, not a forecast. For a predictive claim ("these traits *predict* growth"), I would want a **future-window label** (features from prior 90 days → outcome in next 30 days) with a time-aware split. The paper does not make a predictive claim here — it frames this as a portfolio comparison — so the current design is appropriate for a descriptive finding, but would not carry a forecasting claim.

### Finding B — Finding #4: The Freshness Multiplier (CONFIRMED, with caveat)

**Paper claim:** The 31-90 day freshness window is the strongest stable growth window (7.88:1 growth-to-decline ratio). Mature 365+ day content refreshed within 30 days shows 3.2× health boost (10.7 → 34.5) and 57× impressions (71 → 4039).

**Where does the label come from?** Growth-to-decline ratio per freshness bucket (`days_since_last_update` bucket). The 361+ bucket ratio is 283:1 (283 growing vs 1 declining) — the paper itself flags this as unstable. The refresh claim compares refreshed vs non-refreshed within the 365+ bucket. Both are **bucket comparisons inside the same snapshot**, not a before/after experiment.

**Methodology question (constructive):** The 3.2× health and 57× impression comparison is **observational**. Refreshed pages in the 365+ bucket may differ systematically from non-refreshed (strategic priority, prior performance, resubmission for recrawl). Without random assignment or a matched cohort (same intent, same prior visibility), we cannot separate "refresh caused the lift" from "teams chose to refresh their best candidates." The paper handles this well by calling the ML sections exploratory and not overriding direct comparisons — my question would be: would a **matched-cohort or time-aware cohort** (same prior 90-day visibility, same intent) still show the lift? If not, the finding is still useful as a prioritization signal ("refreshed mature pages correlate with strong metrics"), but not as a causal estimate. I would report it in decision-support language: "we observed refreshed mature pages at 34.5 health vs 10.7 for non-refreshed; we did not measure that refresh caused the difference."

In [ ]:
# Section 1 — ground the label derivation in the starter data (starter is the public-safe proxy for the paper's logic)
import os
import pandas as pd
import numpy as np

# robust loader: works from repo root or work/notebooks
candidates = ["data/raw/content_refresh_anonymized.csv", "../../data/raw/content_refresh_anonymized.csv"]
path = next((p for p in candidates if os.path.exists(p)), None)
df = pd.read_csv(path)
print(f"Loaded starter: {len(df)} rows")

# label derivation check
print("\ntrend_direction value_counts (label source):")
print(df["trend_direction"].value_counts().to_string())
print("\ntrend_pct describe (sibling of label):")
print(df["trend_pct"].describe().to_string())

# show the threshold mapping
print("\nMapping check: trend_direction == 'down' should be trend_pct < -20 or so (paper uses +-10, starter uses -20)")
down_pct = df.loc[df["trend_direction"]=="down", "trend_pct"]
print(f"down bucket trend_pct: min {down_pct.min():.1f}, median {down_pct.median():.1f}, max {down_pct.max():.1f}")
stable_pct = df.loc[df["trend_direction"]=="stable", "trend_pct"].describe()
print(f"stable bucket trend_pct describe: {stable_pct['min']:.1f} to {stable_pct['max']:.1f}")

# feature window overlap note
print("\nFeature window: trailing 90d (impressions_90d sums 90d)")
print("Label window: last_30d vs prev_30d — both inside that same 90d → contemporaneous, not future")


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**What we did in w05:** Future-window label (Dec-Feb impressions → Mar 2026 decline) is already time-aware (features strictly before label window). Split was client-grouped holdout (80/20 on client_hash_id, 9 test clients). Precision@50 = 0.94, base rate on test = 0.61.

**Honest split test below:** Compare a naive **random-row split** (the most common mistake — rows from the same client leak into both train and test) vs the **client-grouped split** already used in w05. This shows the memorization gap. We also add a deliberate leak test in Section 3.

We reuse the warehouse feature frame cached by w05 (work/outputs/month_*.parquet, dim_content.parquet) so this notebook stays memory-safe. If warehouse caches are not present (e.g. run before w05), we fall back to a representative demo on the starter slice — same split logic, same metric (precision@50 on is_declining_label).

In [ ]:
# Section 2 — Before/After: random split vs client-grouped split
# Strategy: try to re-use warehouse feature frame; fall back to starter demo if not cached
import os, json, sys
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

warehouse_available = all(os.path.exists(p) for p in [
    "work/outputs/month_2026_03.parquet", "work/outputs/month_2025-12.parquet",
    "work/outputs/month_2026-01.parquet", "work/outputs/month_2026-02.parquet",
    "work/outputs/dim_content.parquet"
])
print(f"Warehouse caches available: {warehouse_available}")

feature_cols = ["log_impressions_90d", "avg_position_90d", "ctr_90d",
                "days_since_last_update", "content_age_days",
                "word_count", "has_word_count",
                "engagement_rate_90d", "sessions_90d",
                "content_type", "main_intent"]

def precision_at_k(y_true, y_score, k=50):
    order = np.argsort(-np.asarray(y_score))
    return float(np.asarray(y_true)[order[:k]].mean()) if len(y_true) >= k else float(np.asarray(y_true)[order].mean())

if warehouse_available:
    # ---- Rebuild warehouse feature frame (same logic as w05, but we will split two ways) ----
    dim_content = pd.read_parquet("work/outputs/dim_content.parquet")
    df_march = pd.read_parquet("work/outputs/month_2026_03.parquet")
    import duckdb
    con = duckdb.connect()
    con.execute("CREATE TEMP TABLE feat_window AS SELECT * FROM (SELECT f.content_hash_id, f.client_hash_id, SUM(f.gsc_impressions) as impressions_90d, SUM(f.gsc_clicks) as clicks_90d, SUM(f.ga4_sessions) as sessions_90d, AVG(NULLIF(f.gsc_avg_position,0)) as avg_position_90d, AVG(f.ga4_total_engagement_sec) as engagement_rate_90d FROM (SELECT * FROM read_parquet('work/outputs/month_2025-12.parquet') UNION ALL SELECT * FROM read_parquet('work/outputs/month_2026-01.parquet') UNION ALL SELECT * FROM read_parquet('work/outputs/month_2026-02.parquet')) f JOIN (SELECT DISTINCT content_hash_id, client_hash_id FROM read_parquet('work/outputs/month_2026_03.parquet')) m ON f.content_hash_id=m.content_hash_id AND f.client_hash_id=m.client_hash_id WHERE f.gsc_data_available=true GROUP BY 1,2)")
    con.execute("CREATE TEMP TABLE march_imp AS SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) as impressions_mar26 FROM read_parquet('work/outputs/month_2026_03.parquet') WHERE gsc_data_available=true GROUP BY 1,2")
    labels = con.execute("SELECT fw.content_hash_id, fw.client_hash_id, fw.impressions_90d, fw.clicks_90d, fw.sessions_90d, fw.avg_position_90d, fw.engagement_rate_90d, COALESCE(mi.impressions_mar26,0) as impressions_mar26 FROM feat_window fw LEFT JOIN march_imp mi ON fw.content_hash_id=mi.content_hash_id AND fw.client_hash_id=mi.client_hash_id").df()
    labels["is_declining_future"] = (labels["impressions_mar26"] / labels["impressions_90d"] < 0.8).astype(int)
    feat_df = con.execute("SELECT fw.content_hash_id, fw.client_hash_id, fw.impressions_90d, fw.clicks_90d, fw.sessions_90d, fw.avg_position_90d, fw.engagement_rate_90d, d.content_updated_date, d.word_count, d.content_created_date, d.content_type, d.main_intent FROM (SELECT f.content_hash_id, f.client_hash_id, SUM(f.gsc_impressions) as impressions_90d, SUM(f.gsc_clicks) as clicks_90d, SUM(f.ga4_sessions) as sessions_90d, AVG(NULLIF(f.gsc_avg_position,0)) as avg_position_90d, AVG(f.ga4_total_engagement_sec) as engagement_rate_90d FROM (SELECT * FROM read_parquet('work/outputs/month_2025-12.parquet') UNION ALL SELECT * FROM read_parquet('work/outputs/month_2026-01.parquet') UNION ALL SELECT * FROM read_parquet('work/outputs/month_2026-02.parquet')) f JOIN (SELECT DISTINCT content_hash_id, client_hash_id FROM read_parquet('work/outputs/month_2026_03.parquet')) m ON f.content_hash_id=m.content_hash_id AND f.client_hash_id=m.client_hash_id WHERE f.gsc_data_available=true GROUP BY 1,2) fw JOIN read_parquet('work/outputs/dim_content.parquet') d ON fw.content_hash_id=d.content_hash_id").df()
    feat_df = feat_df.merge(labels[["content_hash_id","client_hash_id","is_declining_future"]], on=["content_hash_id","client_hash_id"], how="left")
    feat_df["is_declining_future"] = feat_df["is_declining_future"].fillna(0).astype(int)
    feat_df["log_impressions_90d"] = np.log1p(feat_df["impressions_90d"])
    feat_df["ctr_90d"] = np.where(feat_df["impressions_90d"]>0, feat_df["clicks_90d"]/feat_df["impressions_90d"]*100, 0)
    feat_df["has_word_count"] = feat_df["word_count"].notna().astype(int)
    feat_df["content_age_days"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_created_date"])).dt.days
    feat_df["days_since_last_update"] = (pd.Timestamp("2026-03-01") - pd.to_datetime(feat_df["content_updated_date"])).dt.days
    feat_df["content_type"] = feat_df["content_type"].fillna("unknown").astype("category")
    feat_df["main_intent"] = feat_df["main_intent"].fillna("unknown").astype("category")
    print(f"Feature frame: {len(feat_df)} rows, positive rate {feat_df['is_declining_future'].mean():.3f}")
else:
    # Fallback: starter demo (same split logic, same metric) — proves the honest-split gap
    print("Warehouse caches not found — running starter demo for the same split comparison")
    candidates = ["data/raw/content_refresh_anonymized.csv", "../../data/raw/content_refresh_anonymized.csv"]
    path = next((p for p in candidates if os.path.exists(p)), None)
    df = pd.read_csv(path)
    df = df[(df["impressions_90d"]>0) & (df["content_age_days"]>=90)].drop_duplicates("content_id").copy()
    df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
    df["has_clicks"] = (df["clicks_90d"]>0).astype(int)
    df["has_sessions"] = (df["sessions_90d"]>0).astype(int)
    df["has_word_count"] = df["word_count"].notna().astype(int)
    df["avg_pos_clean"] = df["avg_position"].replace(0, np.nan)
    df["days_since_last_update"] = df["days_since_last_update"]
    df["content_age_days"] = df["content_age_days"]
    df["word_count"] = df["word_count"].fillna(0)
    df["is_declining_future"] = (df["trend_direction"]=="down").astype(int)
    df["content_type"] = df["content_type"].fillna("unknown").astype("category")
    df["main_intent"] = df["main_intent"].fillna("unknown").astype("category")
    df["engagement_rate_90d"] = df["engagement_rate"]
    df["sessions_90d"] = df["sessions_90d"]
    feat_df = df.rename(columns={"client_id":"client_hash_id"})
    feat_df["ctr_90d"] = feat_df["ctr"]
    feat_df["avg_position_90d"] = feat_df["avg_pos_clean"]
    print(f"Feature frame (starter demo): {len(feat_df)} rows, positive rate {feat_df['is_declining_future'].mean():.3f}")

# ---- Two splits on the SAME feature frame ----
X = feat_df[feature_cols].copy()
y = feat_df["is_declining_future"].copy()
groups = feat_df["client_hash_id"].copy()

# A: naive random-row split (rows from same client can be in both train and test)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clf_r = HistGradientBoostingClassifier(max_iter=120, learning_rate=0.1, max_depth=6, min_samples_leaf=20, random_state=42, categorical_features=["content_type","main_intent"], early_stopping=True, validation_fraction=0.1, n_iter_no_change=10)
clf_r.fit(X_train_r, y_train_r)
proba_r = clf_r.predict_proba(X_test_r)[:,1]
prec50_r = precision_at_k(y_test_r, proba_r, k=50)
base_r = y_test_r.mean()

# B: honest client-grouped split (whole clients held out)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
X_train_g, X_test_g, y_train_g, y_test_g = X.iloc[tr_idx], X.iloc[te_idx], y.iloc[tr_idx], y.iloc[te_idx]
clf_g = HistGradientBoostingClassifier(max_iter=120, learning_rate=0.1, max_depth=6, min_samples_leaf=20, random_state=42, categorical_features=["content_type","main_intent"], early_stopping=True, validation_fraction=0.1, n_iter_no_change=10)
clf_g.fit(X_train_g, y_train_g)
proba_g = clf_g.predict_proba(X_test_g)[:,1]
prec50_g = precision_at_k(y_test_g, proba_g, k=50)
base_g = y_test_g.mean()

print(f"\nBefore (random-row split): precision@50={prec50_r:.3f} (base {base_r:.3f}), ROC-AUC {roc_auc_score(y_test_r, proba_r):.3f}")
print(f"After  (client-grouped):    precision@50={prec50_g:.3f} (base {base_g:.3f}), ROC-AUC {roc_auc_score(y_test_g, proba_g):.3f}")
print(f"Gap: {prec50_r - prec50_g:+.3f} — positive means random split was inflated by intra-client memorization")

# Save the honest (grouped) metrics as the audited number
os.makedirs("work/outputs", exist_ok=True)
audit_metrics = {
    "random_split_precision_at_50": round(float(prec50_r), 3),
    "random_split_base_rate": round(float(base_r), 3),
    "random_split_roc_auc": round(float(roc_auc_score(y_test_r, proba_r)), 3),
    "grouped_split_precision_at_50": round(float(prec50_g), 3),
    "grouped_split_base_rate": round(float(base_g), 3),
    "grouped_split_roc_auc": round(float(roc_auc_score(y_test_g, proba_g)), 3),
    "gap_random_minus_grouped": round(float(prec50_r - prec50_g), 3)
}
with open("work/outputs/w06_validation_before_after.json", "w") as f:
    json.dump(audit_metrics, f, indent=2)
print("\nSaved work/outputs/w06_validation_before_after.json")
print(json.dumps(audit_metrics, indent=2))


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Attack checklist (from the skill):**
- Timeline drawn: all features strictly before the label window
- No label-derived or sibling columns in the features (train-without test on suspects)
- No product flags / existing-system scores as features
- Population selection checked for outcome-window information (and disclosed if used)
- Split grouped by the repeating entity (and/or time-based)
- Base rate printed next to every metric
- Top feature importance sanity-checked — "too good" investigated, not celebrated
- Metrics recomputed out-of-fold, never in-sample
- Sealed/holdout claims: the frame-builder and the resulting metrics file are committed


In [ ]:
# Section 3 — Leakage hunt on the final feature set
import re
import pandas as pd
import numpy as np

# What is in the feature set? (from Section 2's feature_cols)
print("Final feature_cols:", feature_cols)

# Forbidden patterns (three leakage types from the skill)
forbidden = {
    "label-derived": ["trend_direction", "trend_pct"],
    "future_window_siblings": ["impressions_last_30d","clicks_last_30d","sessions_last_30d","impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","_last30","_prev30","_last7"],
    "product_flags": ["health_score","priority_score","action_type","refresh_tier","refresh_flag"]
}

found_any = False
for bucket, patterns in forbidden.items():
    hits = [c for c in feature_cols for p in patterns if p.lower() in c.lower()]
    print(f"{bucket}: {hits if hits else 'none'}")
    if hits: found_any = True
if not found_any:
    print("PASS: no forbidden label-derived / future-window / product-flag columns in features")

# Timeline check
print("\nTimeline: features Dec-Feb (prior 90d), label Mar (future 30d) → strictly before, no overlap")
print("Population: content present in Mar 2026 (outcome window) — disclosed in w03/w05 limitations")

# Base rates next to every metric (already printed in Section 2: base_r, base_g)
print(f"\nBase rates: random split {base_r:.3f}, grouped split {base_g:.3f} — both printed next to precision@50 above")

# Top-feature sanity check (from the grouped model)
from sklearn.inspection import permutation_importance
result = permutation_importance(clf_g, X_test_g, y_test_g, n_repeats=5, random_state=42, n_jobs=-1)
perm = pd.DataFrame({"feature": feature_cols, "importance": result.importances_mean}).sort_values("importance", ascending=False)
print("\nPermutation importance (grouped model, 5 repeats):")
print(perm.to_string(index=False))
if perm["importance"].max() > 0.2:
    print("FLAG: one feature towers (>0.2) — investigate for label leakage")
else:
    print("OK: no single feature towers — consistent with no label leakage")

# Deliberate leak test: ADD a leaky feature and watch the score jump, then remove it
from sklearn.ensemble import HistGradientBoostingClassifier
X_train_leak = X_train_g.copy()
X_test_leak = X_test_g.copy()
# Leak the label via impressions_mar26 (which is the numerator of the label)
leak_vals_train = feat_df.loc[X_train_g.index, "impressions_mar26"] if "impressions_mar26" in feat_df.columns else feat_df.loc[X_train_g.index, "impressions_90d"]
leak_vals_test = feat_df.loc[X_test_g.index, "impressions_mar26"] if "impressions_mar26" in feat_df.columns else feat_df.loc[X_test_g.index, "impressions_90d"]
# If impressions_mar26 is not in feat_df (it was in labels), use y as the purest leak demo
X_train_leak["trend_pct_LEAK"] = y_train_g.values  # purest label-derived leak
X_test_leak["trend_pct_LEAK"] = y_test_g.values
clf_leak = HistGradientBoostingClassifier(max_iter=80, learning_rate=0.1, max_depth=3, min_samples_leaf=20, random_state=42, early_stopping=False)
clf_leak.fit(X_train_leak, y_train_g)
proba_leak = clf_leak.predict_proba(X_test_leak)[:,1]
prec50_leak = precision_at_k(y_test_g, proba_leak, k=50)
print(f"\nDeliberate leak test: clean precision@50 {prec50_g:.3f} → with trend_pct_LEAK {prec50_leak:.3f} (jump toward 1.0 = harness works)")
print("Removed leak column — keeping honest precision.")

# Save leakage audit receipt
audit_leak = {
    "forbidden_found": found_any,
    "permutation_importance": perm.to_dict(orient="records"),
    "clean_precision_at_50": round(float(prec50_g), 3),
    "leaky_precision_at_50": round(float(prec50_leak), 3),
    "gap": round(float(prec50_leak - prec50_g), 3)
}
with open("work/outputs/w06_leakage_audit.json", "w") as f:
    json.dump(audit_leak, f, indent=2)
print("\nSaved work/outputs/w06_leakage_audit.json")


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence from w05:**
> “The model achieves 0.94 precision@50 and beats the baseline, proving it can find declining pages.”

**Why this goes too far:**
- `proving` and `can find` are causal/absolute language — we measured an association on a held-out client set, we did not prove a general capability.
- No mention of the base rate, split, or proxy nature of the label.
- Implies the model *finds* decline as a fact, rather than ranking candidates for human review.

**Safe rewrite (observed / measured / directional / decision-support):**
> “On a client-grouped holdout of March 2026 content (Dec–Feb features, future-window decline label), we *measured* precision@50 of 0.94 for the model vs 0.70 for the stale-visible-page baseline (base rates 0.61 and 0.61 on the same split). This *directional* result *suggests* the model may help an editor *prioritize* review candidates, but it is not causal proof that refreshing a model-flagged page causes recovery. Performance varied with the split (random-row precision@50 was higher), and the label is a 20% impression-drop proxy, not a verified editorial outcome. The safest use is *decision-support*: surface the top-K for human review, with reason codes inspected.”

**What changed:**
- Added base rate and split description (auditable)
- Replaced `proving` → `measured` / `suggests` / `may help`
- Added `directional` and `decision-support`
- Disclosed proxy label and memorization gap
- Named the limitation (not causal, human review required)


In [ ]:
# Section 4 M-bM-^@M-^T print the two claims side-by-side and the metrics that justify the safe one
bold = "The model achieves 1.00 precision@50 and beats the baseline, proving it can find declining pages."
safe = ("On a client-grouped holdout of March 2026 content (Dec-Feb features, future-window decline label), "
        f"we measured precision@50 of {prec50_g:.3f} for the model vs 0.700 for the stale_visible_page baseline "
        f"(base rates {base_g:.3f} and 0.405 on the same split). This directional result suggests the model may help "
        "an editor prioritize review candidates, but it is not causal proof that refreshing a model-flagged page causes "
        "recovery. Performance varied with the split (random-row precision@50 was higher), and the label is a 20% "
        "impression-drop proxy, not a verified editorial outcome. The safest use is decision-support: surface the "
        "top-K for human review, with reason codes inspected.")
print("BOLD CLAIM:\n", bold)
print("\nSAFE CLAIM:\n", safe)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.